### Esercizi SQL Esteso

Sono date le seguenti tabelle:

**Sensore** (*idSensore*, Citta, Regione) 

**Data** (*idData*, Data, Mese, Trimestre, Anno) 

**Ente** (*idEnte*, Ente) 

**Misurazioni** (*idSensore*, idData, idEnte, NumMisurazioni) 


1. Per l'anno 2020, separatamente per mese, città ed ente, calcolare: 

    - il numero totale di misurazioni effettuate 

    - la frazione di misurazioni effettuate rispetto al totale di tutto l'anno per tutte le città e per tutti gli enti 

    - la frazione di misurazioni effettuate rispetto a quelle dell'ente che ha effettuato il maggior numero di misurazioni in una città della regione considerata, separatamente per ogni mese 


In [ ]:
SELECT D.Mese, S.Città, E.Ente,
    SUM(M.NumMisurazioni),
    SUM(M.NumMisurazioni) / SUM(SUM(M.NumMisurazioni)) OVER(),
    SUM(M.NumMisurazioni) / SUM(SUM(M.NumMisurazioni)) OVER(
        PARTITION BY D.Mese, S.Città
    )
FROM Misurazioni M
JOIN Sensore S ON S.idSensore = M.idSensore
JOIN Data D ON D.idData = M.idData
JOIN Ente E ON E.idEnte = M.idEnte
WHERE D.Anno = 2020
GROUP BY D.Mese, S.Città, S.Regione, E.Ente ; 

2. Separatamente per città e mese, calcolare: 

    - il numero totale di misurazioni effettuate 

    - la percentuale di misurazioni rispetto al totale mensile della regione 

    - la classifica mensile (rank) delle città con più misurazioni (posizione 1 per la città con più misurazioni, separatamente per ogni mese) 

    - la media del numero di misurazioni mensili effettuate nel mese corrente e nei due 
    mesi precedenti 


In [ ]:
SELECT S.Città, D.Mese, 
    SUM(M.NumMisurazioni),
    100 * SUM(M.NumMisurazioni) / SUM(SUM(M.NumMisurazioni)) OVER(
        PARTITION BY S.Regione, D.Mese
    ),
    RANK() OVER(
        PARTITION BY D.Mese
        ORDER BY SUM(M.NumMisurazioni) DESC
    ),
    AVG(SUM(M.NumMisurazioni)) OVER(
        PARTITION BY S.Città
        ORDER BY D.Mese ASC
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    )
FROM Misurazioni M
JOIN Sensore S ON S.idSensore = M.idSensore
JOIN Data D ON D.idData = M.idData
WHERE
GROUP BY S.Città, S.Regione, D.Mese

3. Separatamente per regione, ente e trimestre, calcolare:

    - il numero medio di misurazioni mensili effettuate;

    - la percentuale di misurazioni rispetto al totale trimestrale effettuato dallo stesso ente sull'intero territorio nazionale;

    - il numero cumulativo di misurazioni al trascorrere dei trimestri, separatamente per anno, regione ed ente;

    - la classifica (rank) delle regioni con più misurazioni (posizione 1 per la regione con più misurazioni), calcolata separatamente per ciascun ente e trimestre.

In [ ]:
SELECT S.Regione, E.Ente, D.Trimestre,
    1.0 * SUM(M.NumMisurazioni) / COUNT(DISTINCT D.Mese),
    100 * SUM(M.NumMisurazioni) / SUM(SUM(M.NumMisurazioni)) OVER(
        PARTITION BY E.Ente, D.Trimestre
    ),
    SUM(SUM(M.NumMisurazioni)) OVER(
        PARTITION BY D.Anno, S.Regione, E.Ente
        ORDER BY D.Trimestre
        ROWS UNBOUNDED PRECEDING
    ),
    RANK() OVER(
        PARTITION BY E.Ente, D.Trimestre, D.Anno
        ORDER BY SUM(M.NumMisurazioni) DESC
    )
FROM Misurazioni M
JOIN Sensore S ON S.idSensore = M.idSensore
JOIN Ente E ON E.idEnte = M.idEnte
JOIN Data D ON D.idData = M.idData
GROUP BY S.Regione, E.Ente, D.Trimestre, D.Anno

4. Considerare le seguenti 3 query di interesse frequenti sul Data Warehouse:

    Query A: Per l'ente 'ARPA', visualizzare il numero totale di misurazioni e la media mensile, separatamente per città e trimestre.

    Query B: Per i sensori situati nella regione 'Piemonte', calcolare il numero cumulativo di misurazioni al trascorrere dei trimestri, separatamente per anno ed ente.

    Query C: Separatamente per anno e regione, visualizzare la percentuale di misurazioni rispetto al totale annuale nazionale.

a) Considerando le informazioni precedenti, definire una vista materializzata (ViewMisurazioni) con **CREATE MATERIALIZED VIEW**, in modo da ridurre il tempo di risposta delle query di interesse sopra riportate. 

In [ ]:
CREATE MATERIALIZED VIEW ViewMisurazioni
BUILD IMMEDIATE
REFRESH FAST ON COMMIT
AS
SELECT S.Città, S.Regione, D.Mese, D.Trimestre, D.Anno, E.Ente, SUM(M.NumMisurazioni) AS TotMisurazioni
FROM Misurazioni M
JOIN Sensore S ON S.idSensore = M.idSensore
JOIN Ente E ON E.idEnte = M.idEnte
JOIN Data D ON D.idData = M.idData
GROUP BY S.Città, S.Regione, D.Trimestre, D.Mese, D.Anno, E.Ente

b) Insieme minimale: indicare l'insieme minimale di attributi che identifica univocamente le tuple della vista ViewMisurazioni.

Insieme minimale = {Città, Mese, Ente}

c) Si ipotizzi che la gestione della vista materializzata (tabella derivata) sia svolta mediante trigger. Scrivere il trigger per propagare le modifiche alla vista materializzata ViewMisurazioni in caso di inserimento di un nuovo record nella tabella dei fatti Misurazioni. 

In [ ]:
CREATE OR REPLACE TRIGGER AggiornaViewMisurazioni
AFTER INSERT ON Misurazioni
FOR EACH ROW
DECLARE
    varCittà, varRegione, varMese, varTrimestre, varAnno, varEnte VARCHAR(20);
    N INTEGER;
BEGIN
    SELECT Città, Regione
    INTO varCittà, varRegione
    FROM Sensore 
    WHERE idSensore = :NEW.idSensore; 

    SELECT Mese, Trimestre, Anno
    INTO varMese, varTrimestre, varAnno
    FROM Data 
    WHERE idData = :NEW.idData; 

    SELECT Ente
    INTO varEnte
    FROM Ente
    WHERE idEnte = :NEW.idEnte;

    SELECT COUNT(*)
    INTO N
    FROM ViewMisurazioni 
    WHERE Città = varCittà
        AND Mese = varMese
        AND Ente = varEnte;
    
    IF (N>0) THEN
        UPDATE ViewMisurazioni
        SET NumMisurazioni = NumMisurazioni + :NEW.NumMisurazioni
        WHERE Città = varCittà
            AND Mese = varMese
            AND Ente = varEnte; 
    ELSE 
        INSERT INTO ViewMisurazioni(
            Città, Regione, Mese, Trimestre, Anno, Ente, TotMisurazioni
        )
        VALUES(
            varCittà, varRegione, varMese, varTrimestre, varAnno, varEnte, :NEW.NumMisurazioni
        );
    END IF;
END;
